# Cost Calculation — Production Pipelines

Measured cost from real token counts in raw model outputs:

- **0518** — Haiku 4.5 + Kimi K2.5, Bedrock batch. Run `9f602b16-b55e-4589-83db-6c000e66ce9d`, 383 clusters.
- **v305** — Sonnet 4.6 single-stage with prompt caching, on-demand. From `experiments_02252026/`, 291 clusters.

Summary metrics: cost per case, cost per 5K opinion-input tokens (= front-of-pipeline content volume; Stage 1 Haiku input for 0518, Sonnet input + cache writes for v305).

## Pricing constants

Bedrock list prices per 1M tokens. Cache write = 1.25× input, cache read = 10% of input (standard Anthropic-on-Bedrock policy). Kimi cached-input rate inferred from the same 10% rule of thumb.

In [1]:
HAIKU_INPUT_PER_M       = 1.00
HAIKU_OUTPUT_PER_M      = 5.00
HAIKU_CACHE_WRITE_PER_M = 1.25
HAIKU_CACHE_READ_PER_M  = 0.10

SONNET_INPUT_PER_M       = 3.00
SONNET_OUTPUT_PER_M      = 15.00
SONNET_CACHE_WRITE_PER_M = 3.75
SONNET_CACHE_READ_PER_M  = 0.30

KIMI_INPUT_PER_M        = 0.60
KIMI_OUTPUT_PER_M       = 3.00
KIMI_CACHED_INPUT_PER_M = 0.06

BATCH_DISCOUNT = 0.50

## 0518 pipeline — token totals + cost

In [2]:
import json
from pathlib import Path
import pandas as pd

RUN_DIR = Path('data/scratch/9f602b16-b55e-4589-83db-6c000e66ce9d')

s1 = {'input': 0, 'cache_create': 0, 'cache_read': 0, 'output': 0, 'records': 0}
with open(RUN_DIR / 'stage1_raw' / 'input.jsonl.out') as f:
    for line in f:
        u = json.loads(line)['modelOutput']['usage']
        s1['input']        += u.get('input_tokens', 0)
        s1['cache_create'] += u.get('cache_creation_input_tokens', 0)
        s1['cache_read']   += u.get('cache_read_input_tokens', 0)
        s1['output']       += u.get('output_tokens', 0)
        s1['records']      += 1

s2 = {'input': 0, 'cached': 0, 'output': 0, 'records': 0}
with open(RUN_DIR / 'stage2_raw' / 'input.jsonl.out') as f:
    for line in f:
        u = json.loads(line)['modelOutput']['usage']
        prompt = u.get('prompt_tokens', 0)
        cached = u.get('prompt_tokens_details', {}).get('cached_tokens', 0)
        s2['input']   += prompt - cached
        s2['cached']  += cached
        s2['output']  += u.get('completion_tokens', 0)
        s2['records'] += 1

def cost_haiku(t, batch=False):
    c = (t['input']*HAIKU_INPUT_PER_M + t['cache_create']*HAIKU_CACHE_WRITE_PER_M
         + t['cache_read']*HAIKU_CACHE_READ_PER_M + t['output']*HAIKU_OUTPUT_PER_M) / 1e6
    return c * (BATCH_DISCOUNT if batch else 1.0)

def cost_kimi(t, batch=False):
    c = (t['input']*KIMI_INPUT_PER_M + t['cached']*KIMI_CACHED_INPUT_PER_M
         + t['output']*KIMI_OUTPUT_PER_M) / 1e6
    return c * (BATCH_DISCOUNT if batch else 1.0)

h_cost = cost_haiku(s1, batch=True)
k_cost = cost_kimi(s2, batch=True)
total_0518 = h_cost + k_cost
n_0518 = 383
input_tokens_0518 = s1['input'] + s1['cache_create'] + s1['cache_read'] + s2['input'] + s2['cached']

print(f'0518 (Haiku + Kimi, batch) \u2014 {n_0518} citing clusters')
print(f'  Stage 1 records: {s1["records"]:,}    Stage 2 records: {s2["records"]:,}')
print(f'  Total input tokens (both stages, all input-side):   {input_tokens_0518:>14,}')
print(f'  Total output tokens (both stages):                  {s1["output"]+s2["output"]:>14,}')
print(f'  Stage 1 batch cost: ${h_cost:>8.4f}')
print(f'  Stage 2 batch cost: ${k_cost:>8.4f}')
print(f'  Total batch cost:   ${total_0518:>8.4f}')

0518 (Haiku + Kimi, batch) — 383 citing clusters
  Stage 1 records: 386    Stage 2 records: 4,118
  Total input tokens (both stages, all input-side):       53,386,406
  Total output tokens (both stages):                       3,384,909
  Stage 1 batch cost: $  5.5802
  Stage 2 batch cost: $ 13.0818
  Total batch cost:   $ 18.6620


## v305 pipeline — token totals + cost

In [3]:
V305_ROOT = Path('/Users/rachel/Desktop/flp/ai-research/experiments_02252026/data/output/v305')
shards = []
for sub in V305_ROOT.iterdir():
    if not sub.is_dir() or sub.name == 'example': continue
    p = sub / 'raw_results.csv'
    if p.exists():
        shards.append(pd.read_csv(p))
v305 = pd.concat(shards, ignore_index=True)

v = {
    'input':        int(v305['input_tokens'].sum()),
    'cache_write':  int(v305['cache_write_input_tokens'].sum()),
    'cache_read':   int(v305['cache_read_input_tokens'].sum()),
    'output':       int(v305['output_tokens'].sum()),
    'records':      len(v305),
}

def cost_sonnet(t, batch=False):
    c = (t['input']*SONNET_INPUT_PER_M + t['cache_write']*SONNET_CACHE_WRITE_PER_M
         + t['cache_read']*SONNET_CACHE_READ_PER_M + t['output']*SONNET_OUTPUT_PER_M) / 1e6
    return c * (BATCH_DISCOUNT if batch else 1.0)

v_cost_actual = cost_sonnet(v, batch=False)  # v305 was on-demand
v_cost_batch  = cost_sonnet(v, batch=True)
n_v305 = v305['citing_cluster_id'].nunique()
input_tokens_v305 = v['input'] + v['cache_write'] + v['cache_read']

print(f'v305 (Sonnet 4.6 single-stage, on-demand) \u2014 {n_v305} citing clusters')
print(f'  Records: {v["records"]:,}')
print(f'  Total input tokens (input + cache write + cache read): {input_tokens_v305:>14,}')
print(f'  Total output tokens:                                   {v["output"]:>14,}')
print(f'  Cost (on-demand, actual): ${v_cost_actual:>8.4f}')
print(f'  Cost (batch hypothetical): ${v_cost_batch:>8.4f}')

v305 (Sonnet 4.6 single-stage, on-demand) — 291 citing clusters
  Records: 291
  Total input tokens (input + cache write + cache read):      7,386,407
  Total output tokens:                                        2,588,902
  Cost (on-demand, actual): $ 60.4070
  Cost (batch hypothetical): $ 30.2035


## Summary — cost per case + cost per 5,000 input tokens

In [4]:
def summary_row(label, n, opinion_in_tok, out_tok, total):
    """opinion_in_tok = the input-side tokens that represent one-pass-through-the-pipeline.
    For 0518 this is Stage 1 (Haiku) input only — Stage 2 re-reads section text per group.
    For v305 this is input + cache_write — the fresh content the single Sonnet call saw.
    """
    return {
        'pipeline': label,
        'clusters': n,
        'opinion_input_tokens': opinion_in_tok,
        'output_tokens': out_tok,
        'total_cost_usd': total,
        'cost_per_case_usd': total / n,
        'cost_per_5k_opinion_input_usd': total / opinion_in_tok * 5000,
        'cost_per_10B_opinion_input_usd': total / opinion_in_tok * 10_000_000_000,
    }

opinion_in_0518 = s1['input']                 # Stage 1 Haiku input
opinion_in_v305 = v['input'] + v['cache_write']  # fresh content seen by Sonnet

summary = pd.DataFrame([
    summary_row('0518 Haiku+Kimi (batch)',         n_0518, opinion_in_0518, s1['output']+s2['output'], total_0518),
    summary_row('v305 Sonnet (on-demand, actual)', n_v305, opinion_in_v305, v['output'],               v_cost_actual),
    summary_row('v305 Sonnet (batch hypothetical)',n_v305, opinion_in_v305, v['output'],               v_cost_batch),
]).set_index('pipeline').round(4)
summary

,clusters,opinion_input_tokens,output_tokens,total_cost_usd,cost_per_case_usd,cost_per_5k_opinion_input_usd,cost_per_10B_opinion_input_usd
pipeline,,,,,,,
0518 Haiku+Kimi (batch),383,7237898,3384909,18.6620,0.0487,0.0129,25783.7215
"v305 Sonnet (on-demand, actual)",291,5612659,2588902,60.4070,0.2076,0.0538,107626.4132
v305 Sonnet (batch hypothetical),291,5612659,2588902,30.2035,0.1038,0.0269,53813.2066


## Demo extrapolation (4,125 clusters)

In [5]:
DEMO_N = 4125
extrap = summary[['cost_per_case_usd']].copy()
extrap['demo_total_usd_4125'] = (extrap['cost_per_case_usd'] * DEMO_N).round(2)
extrap

,cost_per_case_usd,demo_total_usd_4125
pipeline,,
0518 Haiku+Kimi (batch),0.0487,200.89
"v305 Sonnet (on-demand, actual)",0.2076,856.35
v305 Sonnet (batch hypothetical),0.1038,428.18


## Notes

- **0518's total input-side volume is ~7× v305's** (53.4M vs 7.4M) because Stage 2 (Kimi) makes ~10.75 calls per cluster, each reading section context + prompt + citation list. Stage 1 reads each opinion once. v305 is single-stage so the opinion is read once per cluster. The two-stage architecture trades higher token throughput for cheaper per-token rates (Haiku is 1/3 of Sonnet's input; Kimi cached input ~10% of normal); net per-case cost lands lower.
- **v305 was on-demand** — cache-write tokens dominate its bill. Batch would be ~50% cheaper.
- **Prior proxy estimates** ($0.024/case batch from 0407 9-case; $75–99 batch for 4,125 from 0501) were ~2× too low. Measured 0518 is $0.047/case batch → ~$201 for 4,125.